In [ ]:
"""
SELF-PRUNING NEURAL NETWORK (FIXED + β ANNEALING)
CIFAR-10 | Hard Concrete-style sigmoid pruning
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# =========================================================
# CONFIG
# =========================================================
EPOCHS = 20
BATCH_SIZE = 128
LR = 3e-4
LAM = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================================================
# β ANNEALING (IMPORTANT FIX)
# =========================================================
def beta_schedule(epoch, max_epoch):
    # starts smooth → becomes sharp (critical for pruning)
    return 1.0 + 9.0 * (epoch / max_epoch)

# =========================================================
# PRUNABLE LAYER
# =========================================================
class PrunableLinear(nn.Module):
    def __init__(self, in_f, out_f):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_f, in_f) * 0.02)
        self.bias = nn.Parameter(torch.zeros(out_f))

        # gate scores (important: start slightly negative → encourages pruning)
        self.gate_scores = nn.Parameter(torch.randn(out_f, in_f) - 2.0)

    def gates(self, beta):
        return torch.sigmoid(beta * self.gate_scores)

    def forward(self, x, beta):
        g = self.gates(beta)
        w = self.weight * g
        return F.linear(x, w, self.bias)

    def sparsity(self, beta, thr=0.3):
        g = self.gates(beta).detach()
        return (g < thr).float().mean().item()

# =========================================================
# MODEL
# =========================================================
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = PrunableLinear(3072, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 10)

    def forward(self, x, beta):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x, beta))
        x = F.relu(self.fc2(x, beta))
        x = self.fc3(x, beta)
        return x

    def sparsity(self, beta):
        return (
            self.fc1.sparsity(beta) +
            self.fc2.sparsity(beta)
        ) / 2

# =========================================================
# DATA
# =========================================================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=transform)
test = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test, batch_size=BATCH_SIZE)

# =========================================================
# TRAIN / EVAL
# =========================================================
def train():
    model = Net().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):

        beta = beta_schedule(epoch, EPOCHS)

        print("\nEpoch", epoch+1, "| β=", round(beta, 4))

        model.train()
        correct = total = 0

        for i, (x, y) in enumerate(train_loader):

            x, y = x.to(device), y.to(device)

            out = model(x, beta)

            loss_cls = F.cross_entropy(out, y)

            # sparsity loss (VERY IMPORTANT FIX)
            sparsity_loss = 0
            for m in model.modules():
                if isinstance(m, PrunableLinear):
                    sparsity_loss += m.gates(beta).mean()

            loss = loss_cls + LAM * sparsity_loss

            opt.zero_grad()
            loss.backward()
            opt.step()

            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

            # PRINT EVERY 50 BATCHES (IMPORTANT FIX)
            if i % 50 == 0:
                acc = 100 * correct / total
                sp = model.sparsity(beta)
                print(f"Batch {i}/{len(train_loader)} | "
                      f"Loss {loss.item():.3f} | "
                      f"Acc {acc:.2f}% | "
                      f"Sparsity {sp*100:.2f}%")

        # =====================================================
        # TEST
        # =====================================================
        model.eval()
        correct = total = 0

        with torch.no_grad():
            for x, y in test_loader:
                x, y = x.to(device), y.to(device)
                out = model(x, beta)
                pred = out.argmax(1)
                correct += (pred == y).sum().item()
                total += y.size(0)

        test_acc = 100 * correct / total
        sparsity = model.sparsity(beta) * 100

        print("\n──── SUMMARY ────")
        print(f"Test Accuracy: {test_acc:.2f}%")
        print(f"Sparsity     : {sparsity:.2f}%")
        print("------------------")

# =========================================================
# RUN
# =========================================================
train()

Device: cpu

Epoch 1 | β= 1.0
Batch 0/391 | Loss 2.303 | Acc 7.81% | Sparsity 87.62%
Batch 50/391 | Loss 2.217 | Acc 19.39% | Sparsity 87.59%
Batch 100/391 | Loss 2.085 | Acc 20.64% | Sparsity 87.50%
Batch 150/391 | Loss 2.057 | Acc 22.41% | Sparsity 87.44%
Batch 200/391 | Loss 1.945 | Acc 23.94% | Sparsity 87.39%
Batch 250/391 | Loss 1.896 | Acc 25.40% | Sparsity 87.35%
Batch 300/391 | Loss 1.830 | Acc 26.63% | Sparsity 87.31%
Batch 350/391 | Loss 1.731 | Acc 27.60% | Sparsity 87.29%

──── SUMMARY ────
Test Accuracy: 35.33%
Sparsity     : 87.27%
------------------

Epoch 2 | β= 1.45
Batch 0/391 | Loss 1.972 | Acc 36.72% | Sparsity 91.99%
Batch 50/391 | Loss 1.921 | Acc 33.04% | Sparsity 91.90%
Batch 100/391 | Loss 1.951 | Acc 33.75% | Sparsity 91.83%
Batch 150/391 | Loss 1.794 | Acc 34.12% | Sparsity 91.76%
Batch 200/391 | Loss 1.828 | Acc 34.39% | Sparsity 91.72%
Batch 250/391 | Loss 1.688 | Acc 34.69% | Sparsity 91.69%
Batch 300/391 | Loss 1.734 | Acc 35.07% | Sparsity 91.67%
Batch 